In [47]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

import functions
import pandas as pd
from PIL import Image
import os

In [48]:
scaler = StandardScaler()
WINDOW_SIZE=25
STRIDE=1
train_ds, test_ds, test_labels = functions.load_preprocessed_SMD_windows(1, 1, scaler=scaler, window_size=WINDOW_SIZE, frame_step=STRIDE)

In [ ]:
class MLPProjector(keras.layers.Layer):
    def __init__(self, hidden_units=32, out_dim=64, use_bias=True, leaky_alpha=0.01, **kwargs):
        super().__init__(**kwargs)

        self.model = keras.Sequential([
            keras.layers.Dense(hidden_units),
            keras.layers.BatchNormalization(),
            keras.layers.ReLU(),
            keras.layers.Dense(out_dim),
        ])
        
    def call(self, x, training=False):
        """
        Project the context into feature space of out_dim. Note: project then pool approached is used

        Args:
            x (Tensor): (B, T, F)
            training (bool, optional): training status flag. Defaults to False.

        Returns:
            z (Tensor): Main projection
            z_prime (Tensor) : 
        """
        return self.model(x, training=training)

In [ ]:
class MLPProjector(keras.layers.Layer):
    def __init__(self, hidden_units=32, out_dim=64, use_bias=True, leaky_alpha=0.01, **kwargs):
        super().__init__(**kwargs)

        self.model = keras.Sequential([
            keras.layers.Dense(hidden_units),
            keras.layers.BatchNormalization(),
            keras.layers.ReLU(),
            keras.layers.Dense(out_dim),
        ])
        
    def call(self, x, training=False):
        return self.model(x, training=training)

class StackedEncoder(keras.layers.Layer):
    def __init__(self, num_hidden_layers=2, bottleneck=False, hidden_units=64, latent_dim=10, dropout_rate=0.2, kernel_regularizer=None, recurrent_regularizer=None, 
                dropout=0,  **kwargs):
        super().__init__(**kwargs) 
        self.latent_output_shape = latent_dim 
        self.layers = []
        for i in range(num_hidden_layers):
            hu = int(hidden_units / (i+1)) if bottleneck else hidden_units
            self.layers.append(
                keras.layers.LSTM(hu, return_sequence=True, kernel_initializer=tf.keras.initializers.GlorotUniform(), dropout=dropout_rate,
                                  recurrent_initializer=tf.keras.initializers.Orthogonal(), kernel_regularizer=kernel_regularizer,
                                  recurrent_regularizer=recurrent_regularizer)
            )
        self.layers.append(
                keras.layers.LSTM(latent_dim, return_sequence=False, kernel_initializer=tf.keras.initializers.GlorotUniform(), dropout=dropout_rate,
                                  recurrent_initializer=tf.keras.initializers.Orthogonal(), kernel_regularizer=kernel_regularizer,
                                  recurrent_regularizer=recurrent_regularizer)
            )
        
        self.model = keras.Sequential(self.layers)
        
    def call(self, x, training=False):
        return self.model(x, training=training)
    
class StackedDecoder(keras.layers.Layer):
    def __init__(self, out_dim, timesteps, num_hidden_layers=2, bottleneck=False, hidden_units=64, dropout_rate=0.2, kernel_regularizer=None, recurrent_regularizer=None, 
            dropout=0,  **kwargs):
        super().__init__(**kwargs) 
        assert out_dim is not None, "Must specify original non-encoded feature dimension"
        assert timesteps is not None, "Must specify window frame size"
        self.output_dim = out_dim 
        self.layers = [keras.layers.RepeatVector(timesteps)]
        
        for i in range(num_hidden_layers, 1, -1):
            hu = int(hidden_units / i) if bottleneck else hidden_units
            self.layers.append(
                keras.layers.LSTM(hu, return_sequence=True, kernel_initializer=tf.keras.initializers.GlorotUniform(), dropout=dropout_rate,
                                  recurrent_initializer=tf.keras.initializers.Orthogonal(), kernel_regularizer=kernel_regularizer,
                                  recurrent_regularizer=recurrent_regularizer)
            )
        self.layers.append(keras.layers.TimeDistributed(
                keras.layers.Dense(out_dim, activation='linear')))    
        self.model = keras.Sequential(self.layers)
        
    def call(self, x, training=False):
        return self.model(x, training=training)

In [ ]:
class LSTM_CBCL(keras.Model):
    def __init__(self, timesteps, n_features, latent_dim, num_layers=2, bottleneck=True, hidden_units = 64,
                 dropout_rate=0.2, kernel_regularizer=False, batch_size=128, recurrent_regularizer=True):
        super().__init__()

        self.encoder = StackedEncoder(num_hidden_layers=num_layers, bottleneck=bottleneck, hidden_units=hidden_units,
                                      latent_dim=latent_dim, dropout_rate=dropout_rate, 
                                      kernel_regularizer=tf.keras.regularizers.L2(1e-4) if kernel_regularizer else None, 
                                      recurrent_regularizer=tf.keras.regularizers.L2(1e-4) if recurrent_regularizer else None)
        
        self.decoder = StackedDecoder(n_features, timesteps, num_hidden_layers=num_layers, bottleneck=bottleneck, 
                                      hidden_units=hidden_units, dropout_rate=dropout_rate, 
                                      kernel_regularizer=tf.keras.regularizers.L2(1e-4) if kernel_regularizer else None, 
                                      recurrent_regularizer=tf.keras.regularizers.L2(1e-4) if recurrent_regularizer else None)
        
        self.projector_head = 
        